[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Introducción a *PyArrow*.

## ¿Qué es *PyArrow*?

### Historia y contexto.

[*Apache Arrow*](https://arrow.apache.org/) es un proyecto de código abierto que define un formato estandarizado para datos columnares en memoria. Fue anunciado en 2016 como una iniciativa colaborativa entre múltiples organizaciones del ecosistema de datos.

Uno de sus principales impulsores fue [**Wes McKinney**](https://wesmckinney.com/), creador de *Pandas*. McKinney había identificado de primera mano las limitaciones de su propia biblioteca y las documentó en el influyente artículo [*"Apache Arrow and the '10 Things I Hate About pandas'"*](https://wesmckinney.com/blog/apache-arrow-pandas-internals/) (2017). En él argumentaba que *Pandas* sufría de problemas estructurales difíciles de resolver sin un rediseño profundo del formato de memoria subyacente: consumo excesivo de RAM, copias innecesarias de datos y falta de interoperabilidad entre herramientas. *Apache Arrow* fue su respuesta a esos problemas.

El proyecto fue diseñado para resolver tres limitaciones comunes en el ecosistema de datos:

1. **Heterogeneidad de formatos:** cada biblioteca tenía su propia representación interna (*NumPy*, *Pandas*, *Spark*, etc.)
2. **Ineficiencia de memoria:** necesidad de copias repetidas al pasar datos entre sistemas
3. **Cuellos de botella en I/O:** lectura y escritura lenta de formatos binarios

### Ventajas de *Arrow*.

| Ventaja | Descripción |
| :--- | :--- |
| **Memoria eficiente** | Formato columnar *zero-copy* |
| **Interoperabilidad** | Compatible con múltiples lenguajes (*C++*, *Java*, *Python*, *Rust*…) |
| **Velocidad** | Operaciones vectorizadas que aprovechan las instrucciones SIMD del procesador |
| **Compresión** | Soporta múltiples esquemas de compresión |
| **Estandarización** | Formato universal para datos complejos |
> **Zero-copy** significa que al transferir datos entre dos bibliotecas (por ejemplo, de *PyArrow* a *Polars*), no se crea ninguna copia del bloque de memoria: ambas bibliotecas apuntan a la misma región de *RAM*. Esto elimina el coste de tiempo y memoria de duplicar datos, algo habitual en el ecosistema anterior a *Arrow*.

### *PyArrow* como *backend* de *Pandas*.

Desde **Pandas 2.0**, es posible usar *PyArrow* como *backend* para *DataFrames*, reemplazando el antiguo sistema de tipos *NumPy* con tipos más eficientes y flexibles.

*Polars*, la biblioteca de análisis de datos de alto rendimiento que se verá en los próximos capítulos, usa el formato *Arrow* como su formato de memoria nativo. Entender *PyArrow* es por tanto la base para comprender de dónde proviene el rendimiento de *Polars*.

* La siguiente celda instala *PyArrow* en el entorno actual en caso de que no esté disponible.

In [ ]:
%pip install pyarrow

* La siguiente celda importa `pyarrow` con el alias convencional `pa`, junto con `pandas`, `numpy` y el módulo `pyarrow.compute` como `pc`.

In [ ]:
import pyarrow as pa
import pyarrow.compute as pc
import pandas as pd
import numpy as np
import os

## *Arrays* de *PyArrow*.

Un *array* de *PyArrow* es una colección homogénea de valores de un único tipo, almacenada eficientemente en memoria.

### Creación de *arrays*.

Se pueden crear *arrays* a partir de:
* Listas de *Python*
* *Arrays* de *NumPy*
* Iteradores

**Ejemplos:**

* La siguiente celda crea un *array* simple desde una lista de *Python*. *PyArrow* infiere automáticamente el tipo de datos.

In [ ]:
arr_simple = pa.array([1, 2, 3, 4, 5])
arr_simple

* La siguiente celda inspecciona la propiedad `.type` del *array* para ver el tipo de datos inferido automáticamente.

In [ ]:
arr_simple.type

* La siguiente celda crea un array con valores nulos (`None`). *PyArrow* los representa internamente con un *bitmask* eficiente y expone la propiedad `null_count` para detectarlos sin recorrer todos los elementos.

In [ ]:
arr_nulls = pa.array([1, 2, None, 4, 5])
arr_nulls

* La siguiente celda evalúa `null_count > 0` para verificar si el *array* contiene valores nulos.

In [ ]:
arr_nulls.null_count > 0

* La siguiente celda crea *arrays* con tipos de datos explícitos usando el parámetro `type=`. *PyArrow* ofrece un sistema de tipos rico que incluye enteros de distinto ancho, flotantes, cadenas y muchos otros.

In [ ]:
arr_int32 = pa.array([1, 2, 3], type=pa.int32())
arr_int32

* La siguiente celda crea un *array* de tipo `float64` con valores decimales.

In [ ]:
arr_float64 = pa.array([1.5, 2.5, 3.5], type=pa.float64())
arr_float64

* La siguiente celda crea un *array* de tipo `string`.

In [ ]:

arr_string = pa.array(['apple', 'banana', 'cherry'], type=pa.string())

* La siguiente celda convierte un *array* de *NumPy* a *PyArrow* con `pa.array()`. La conversión es directa y *zero-copy* cuando los tipos son compatibles.

In [ ]:
pa.array(np.array([10, 20, 30, 40, 50]))

## El módulo `pyarrow.compute`.

El módulo `pyarrow.compute` ofrece una amplia gama de funciones para manipular *arrays* de manera eficiente. Estas funciones están optimizadas para aprovechar las capacidades de hardware modernas, como las instrucciones SIMD, lo que permite realizar operaciones en grandes conjuntos de datos de manera rápida.

Por convención, se importa el módulo `pyarrow.compute` como `pc` para acceder a las funciones de computación.

```
pc.<función>(<array>, <argumentos>)
```
 Donde:

* `<función>` es el nombre de la operación (e.g., `add`, `mean`, `filter`).
* `<array>` es el *array* de entrada sobre el que se realizará la operación.
* `<argumentos>` son parámetros adicionales necesarios para la operación (e.g., otro *array* para operaciones binarias, o una función de filtro).

[Documentación de `pyarrow.compute`](http://arrow.apache.org/docs/python/compute.html)

### Funciones de computación comunes.

> **Nota:** *Polars* —que se estudiará en el siguiente notebook— provee su propia *API* de expresiones para realizar estas operaciones de forma más eficiente y expresiva. Por ello, aquí solo se ejemplifican las funciones más comunes de `pyarrow.compute`.

| Función | Descripción |
| :--- | :--- |
| [`add()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.add.html) | Suma elemento a elemento de dos *arrays* |
| [`mean()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.mean.html) | Calcula la media de un *array* numérico |
| [`filter()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.filter.html) | Filtra un *array* según una condición booleana |
| [`sum()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.sum.html) | Suma todos los elementos de un *array* |
| [`count()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.count.html) | Cuenta el número de elementos no nulos en un *array* |
| [`min()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.min.html) | Encuentra el valor mínimo en un *array* |
| [`max()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.max.html) | Encuentra el valor máximo en un *array* |
| [`is_null()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.is_null.html) | Devuelve un *array* booleano indicando qué elementos son nulos |
| [`cast()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.cast.html) | Convierte un *array* a otro tipo de datos |
| [`sort_indices()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.sort_indices.html) | Devuelve los índices que ordenarían un *array* |
| [`unique()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.unique.html) | Devuelve los valores únicos de un *array* |
| [`quantile()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.quantile.html) | Calcula los cuantiles de un *array* numérico |
| [`top_k()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.top_k.html) | Devuelve los *k* valores más grandes de un *array* |
| [`bottom_k()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.bottom_k.html) | Devuelve los *k* valores más pequeños de un *array* |
| [`concat()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.concat.html) | Concatena varios *arrays* en uno solo |
| [`take()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.take.html) | Toma elementos de un *array* según índices específicos |
| [`if_else()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.if_else.html) | Devuelve elementos de un *array* u otro según una condición booleana |
| [`coalesce()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.coalesce.html) | Devuelve el primer valor no nulo de una lista de *arrays* |
| [`is_in()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.is_in.html) | Devuelve un *array* booleano indicando qué elementos están en otro *array* |
| [`like()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.like.html) | Devuelve un *array* booleano indicando qué elementos de texto coinciden con un patrón |
| [`regexp_match()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.regexp_match.html) | Devuelve un *array* booleano indicando qué elementos de texto coinciden con una expresión regular |
| [`regexp_replace()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.regexp_replace.html) | Reemplaza partes de elementos de texto que coinciden con una expresión regular |
| [`regexp_extract()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.regexp_extract.html) | Extrae partes de elementos de texto que coinciden con una expresión regular |
| [`regexp_extract_all()`](https://arrow.apache.org/docs/python/generated/pyarrow.compute.regexp_extract_all.html) | Extrae todas las partes de elementos de texto que coinciden con una expresión regular |

[Funciones de computación disponibles](https://arrow.apache.org/docs/python/compute.html#available-compute-functions)


**Ejemplos:**

* La siguiente celda crea el *array* de ejemplo sobre el que se aplicarán las funciones de computación.

In [ ]:
arr = pa.array([1, 2, 3, 4, 5, 6])
arr

* La siguiente celda suma 10 a cada elemento del *array* con `pc.add()`.

In [ ]:
pc.add(arr, 10)

* La siguiente celda multiplica cada elemento del *array* por 2 con `pc.multiply()`.

In [ ]:
pc.multiply(arr, 2)

### El método `arr.as_py()`.

El método `as_py()` de un *array* de *PyArrow* convierte el *array* a una representación nativa de *Python*, como una lista o un diccionario, dependiendo del tipo de datos. Esta conversión es útil para inspeccionar los datos o para integrarlos con otras bibliotecas que esperan estructuras de datos estándar de *Python*.

```
arr.función().as_py()
```
Donde:
* `arr` es el *array* de *PyArrow* sobre el que se ha aplicado una función de computación.
* `función()` es la función de computación aplicada al *array* (e.g., `mean()`, `sum()`, `filter()`).
* `as_py()` convierte el resultado de la función de computación a una estructura de datos nativa de *Python* para su fácil manipulación e inspección.

[Documentación de `as_py()`](https://arrow.apache.org/docs/python/arrays.html#as-py)

**Ejemplos:**

* La siguiente celda calcula la suma de todos los elementos del *array* con `pc.sum()` y convierte el resultado a *Python* nativo con `.as_py()`.

In [ ]:
pc.sum(arr).as_py()

* La siguiente celda calcula la media del *array* con `pc.mean()`.

In [ ]:
pc.mean(arr).as_py()

* La siguiente celda obtiene el valor máximo del *array* con `pc.max()`.

In [ ]:
pc.max(arr).as_py()

* La siguiente celda obtiene el valor mínimo del *array* con `pc.min()`.

In [ ]:
pc.min(arr).as_py()

## *Tables* de *PyArrow*.

Una **Table** es una colección de *arrays* columnar con *schema* definido. Es el equivalente de un *dataframe* de *Pandas* y son estructuras de datos fundamentales en *PyArrow* para representar conjuntos de datos tabulares.

Los objetos de tipo **Table** son instancias de `pa.Table` y se crean a partir de un diccionario de *arrays* o de una lista de tuplas, donde cada tupla representa una fila. Cada columna de la tabla tiene un nombre y un tipo de datos asociado, definido por el *schema*.

### Funciones para la creación de *tables*.

Las *tables* de *PyArrow* se pueden crear a partir de diversas fuentes de datos, lo que las hace muy flexibles para integrarse con diferentes formatos y bibliotecas. Algunas de las formas más comunes de crear *tables* incluyen:

* [`pa.Table.from_pandas()`](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.from_pandas): Crea una *table* a partir de un *DataFrame* de *Pandas*. Esta función es útil para aprovechar la eficiencia de *PyArrow* mientras se trabaja con datos que inicialmente están en formato de *Pandas*.
* [`pa.Table.from_arrays()`](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.from_arrays): Crea una *table* a partir de una lista de *arrays* de *PyArrow*. Cada *array* representa una columna de la tabla, y se puede especificar un nombre para cada columna.
* [`pa.table()`](https://arrow.apache.org/docs/python/generated/pyarrow.table.html): Crea una *table* directamente a partir de un diccionario de listas o *arrays*, donde las claves representan los nombres de las columnas.
* [`pa.Table.from_batches()`](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.from_batches): Crea una *table* a partir de una lista de *RecordBatches*, que son bloques de datos tabulares que pueden ser procesados de manera eficiente en memoria.
* [`pyarrow.csv.read_csv()`](https://arrow.apache.org/docs/python/generated/pyarrow.csv.read_csv.html): Lee un archivo *CSV* y devuelve una `Table` de *PyArrow* de manera eficiente, sin pasar por *Pandas*.
* [`pyarrow.parquet.read_table()`](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.read_table.html): Lee un archivo *Parquet* y devuelve una `Table` de *PyArrow*. Es el formato de almacenamiento columnar recomendado para alta performance.

**Ejemplos:**

* La siguiente celda crea una `Table` de *PyArrow* desde un diccionario de listas. El *schema* se infiere automáticamente a partir de los tipos de *Python*.

In [ ]:
data = {
    'id': [1, 2, 3, 4, 5],
    'nombre': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'edad': [25, 30, 35, 28, 32],
    'salario': [50000.0, 60000.0, 75000.0, 55000.0, 70000.0]
}

table = pa.table(data)
print(f"Table:\n{table}")
print(f"\nSchema:\n{table.schema}")

* La siguiente celda accede a columnas individuales de la `Table` usando la sintaxis `table['columna']` y muestra sus dimensiones y nombres de columna.

In [ ]:
print(f"Columna 'nombre': {table['nombre']}")
print(f"\nDimensiones: {table.shape}")
print(f"Nombres de columnas: {table.column_names}")

* La siguiente celda convierte la `Table` a un *DataFrame* de *Pandas* con `.to_pandas()`, mostrando los tipos de datos resultantes.

In [ ]:
df_pandas = table.to_pandas()
print(f"DataFrame (Pandas):\n{df_pandas}")
print(f"\nDtypes:\n{df_pandas.dtypes}")

## El formato *Parquet*.

El formato [*Parquet*](https://parquet.apache.org/) es un formato de almacenamiento columnar optimizado para el rendimiento y la eficiencia, especialmente en entornos de big data. Fue desarrollado por Twitter y Cloudera en 2013 y se ha convertido en uno de los formatos más populares para almacenar grandes volúmenes de datos tabulares.

*Parquet* es uno de los formatos más populares para almacenar datos tabulares debido a varias ventajas clave:
| Ventaja | Descripción |
| :--- | :--- |
| **Eficiencia de almacenamiento** | Al ser un formato columnar, *Parquet* almacena los datos de cada columna juntos, lo que permite una mejor compresión y reducción del espacio en disco. |
| **Rendimiento de lectura** | Al leer solo las columnas necesarias, *Parquet* reduce la cantidad de datos que se deben cargar en memoria, lo que mejora significativamente el rendimiento de lectura, especialmente para conjuntos de datos grandes. |
| **Compatibilidad** | *Parquet* es compatible con una amplia variedad de herramientas y bibliotecas en el ecosistema de datos, incluyendo *Apache Spark*, *Hadoop*, *Pandas*, *PyArrow* y muchas otras. |
| **Soporte para tipos de datos complejos** | *Parquet* soporta tipos de datos complejos como estructuras anidadas, listas y mapas, lo que lo hace adecuado para una amplia gama de casos de uso. |
| **Optimización para big data** | *Parquet* está diseñado para manejar grandes volúmenes de datos, lo que lo hace ideal para entornos de big data y análisis a gran escala. |

Existen algunos otros formatos como `ORC`, `Avro` o `Feather`, pero *Parquet* es el más ampliamente adoptado en la industria debido a su eficiencia y compatibilidad.


## Uso de *schemas*. 

Un *schema* define los nombres y tipos de columnas de una Table. Definirlo explícitamente permite:
* Garantizar tipos de datos correctos al crear o leer datos.
* Detectar errores de tipo antes de procesar grandes volúmenes.
* Documentar la estructura esperada del dato.

*PyArrow* lanza un error inmediato si los datos no coinciden con el schema — sin conversiones silenciosas.

El uso de *schemas* es fundamental para garantizar la integridad y consistencia de los datos, especialmente cuando se trabaja con grandes conjuntos de datos o en entornos de producción donde los errores pueden ser costosos.

### La función `pa.schema()`.

La función `pa.schema()` se utiliza para definir un schema en *PyArrow*. Un *schema* es una estructura que describe los nombres y tipos de columnas de una Table. Al definir un schema explícitamente, se puede garantizar que los datos que se crean o leen cumplan con la estructura esperada, lo que ayuda a detectar errores de tipo antes de procesar grandes volúmenes de datos.

```
pa.schema([
    ('columna1', tipo1),
    ('columna2', tipo2),
    ...
])      
```
Donde:
* `columna1`, `columna2`, ... son los nombres de las columnas que se desean definir en el schema.
* `tipo1`, `tipo2`, ... son los tipos de datos correspondientes a cada columna, que pueden ser tipos primitivos o complejos definidos por *PyArrow*.

[Documentación de `pa.schema()`](https://arrow.apache.org/docs/python/generated/pyarrow.schema.html#pyarrow.schema)

**Ejemplo:**

* La siguiente celda define un *schema* explícito con tipos concretos y crea una `Table` validada contra él.

In [ ]:
schema = pa.schema([
    ('id',      pa.int32()),
    ('nombre',  pa.string()),
    ('edad',    pa.int32()),
    ('salario', pa.float64())
])

* La siguiente celda define el diccionario de datos que se insertará en la `Table`.

In [ ]:
data = {
    'id':      [1, 2, 3],
    'nombre':  ['Alice', 'Bob', 'Charlie'],
    'edad':    [25, 30, 35],
    'salario': [50000.0, 60000.0, 75000.0]
}

* La siguiente celda crea la `Table` validada contra el *schema*, verificando que los tipos sean correctos.

In [ ]:
pa.table(data, schema=schema)

* La siguiente celda define un diccionario con datos inválidos: la columna `id` contiene cadenas de texto en lugar de enteros, en conflicto con el *schema* definido.

In [ ]:
datos_invalidos = {
    'id':['uno', 'dos', 'tres'],
    'nombre':  ['Alice', 'Bob', 'Charlie'],
    'edad':    [25, 30, 35],
    'salario': [50000.0, 60000.0, 75000.0]
    }

* La siguiente celda intenta crear una `Table` con los datos inválidos. *PyArrow* lanzará un `ArrowInvalid` de forma inmediata, sin conversiones silenciosas.

In [ ]:
try:
    pa.table(datos_invalidos, schema=schema)
except pa.ArrowInvalid as e:
    print(f"Error de validación: {e}")

## Tipos de Datos Complejos.

*PyArrow* soporta tipos anidados que permiten representar datos semiestructurados directamente en memoria columnar:

| Tipo | Descripción | Caso de uso |
| :--- | :--- | :--- |
| [`pa.list_()`](https://arrow.apache.org/docs/python/api/array.html#arrow.array) | Lista de valores del mismo tipo | Etiquetas, coordenadas, histórico de valores |
| [`pa.struct()`](https://arrow.apache.org/docs/python/api/array.html#arrow.array) | Registro con campos nombrados y tipados | JSON aplanado, objetos embebidos |

**Ejemplos:**

* La siguiente celda crea un *array* de listas con `pa.list_(pa.int32())`, donde cada elemento es una lista de enteros de longitud variable.

In [ ]:
arr_listas = pa.array(
    [[1, 2, 3], [4, 5], [6, 7, 8, 9]],
    type=pa.list_(pa.int32())
)
arr_listas

* La siguiente celda crea un *array* de *structs* con `pa.struct()`, donde cada elemento es un registro con campos nombrados y tipados, equivalente a un objeto *JSON* en memoria columnar.

In [ ]:
tipo_persona = pa.struct([
    ('nombre', pa.string()),
    ('edad',   pa.int32())
])

arr_structs = pa.array(
    [
        {'nombre': 'Alice',   'edad': 25},
        {'nombre': 'Bob',     'edad': 30},
        {'nombre': 'Charlie', 'edad': 35},
    ],
    type=tipo_persona
)
arr_structs

* La siguiente celda extrae el campo `nombre` de un *array* de *structs* usando `pc.struct_field()`.

In [ ]:
pc.struct_field(arr_structs, 'nombre')

* La siguiente celda extrae el campo `edad` de un *array* de *structs* usando `pc.struct_field()`.

In [ ]:
pc.struct_field(arr_structs, 'edad')

* La siguiente celda calculará la media de las edades en el *array* de *structs* usando `pc.struct_field()` para acceder al campo `edad` y luego `pc.mean()` para calcular la media.

In [ ]:
pc.mean(pc.struct_field(arr_structs, 'edad')).as_py()

## *ChunkedArray* de *PyArrow*.

A diferencia de `pa.Array`, que almacena sus elementos en un único bloque contiguo de memoria, un `ChunkedArray` es una secuencia de *arrays* del mismo tipo organizados en **fragmentos** (*chunks*). Esto permite:

* Procesar conjuntos de datos más grandes que un único bloque de memoria disponible.
* Agregar datos de forma incremental sin copiar el *array* completo.
* Representar las columnas de una `Table` de forma eficiente.

```
pa.chunked_array([array1, array2, ...])
```

Donde:

* `array1`, `array2`, ... son *arrays* de *PyArrow* del mismo tipo que se desean organizar en fragmentos dentro del `ChunkedArray`.

Las columnas de una `Table` de *PyArrow* son siempre `ChunkedArray`, no `Array` simples. *Polars* adopta esta misma estructura internamente para representar cada columna de un `DataFrame`, lo que hace que la conversión entre ambas bibliotecas sea prácticamente *zero-copy*.

**Ejemplos:**

* La siguiente celda crea un `ChunkedArray` a partir de varios *arrays* separados, mostrando el número de fragmentos y la longitud total.

In [ ]:
ca = pa.chunked_array([[1, 2, 3], [4, 5], [6, 7, 8, 9]])
ca

* La siguiente celda mostrará el número de fragmentos en el `ChunkedArray` usando la propiedad `.num_chunks`.

In [ ]:
ca.num_chunks

* La siguiente celda accede a cada fragmento individualmente con `.chunks`.

In [ ]:
print("Chunks individuales:")
for i, chunk in enumerate(ca.chunks):
    print(f"  chunk {i}: {chunk.to_pylist()}")

* La siguiente celda muestra el contenido del *array* `ca` usando `.to_pylist()`.

In [ ]:
ca.combine_chunks().to_pylist()

## Integración con *Pandas*.

*PyArrow* puede ser usado como backend en *Pandas* para mejorar rendimiento y flexibilidad de tipos.

Los tipos de datos de *PyArrow* se mapean a tipos de *Pandas* de la siguiente manera:

| Tipo de *PyArrow* | Tipo de *Pandas* |
| :--- | :--- |
| `pa.int32()` | `int32` |
| `pa.float64()` | `float64` |
| `pa.string()` | `object` (cadena de texto) |
| `pa.list_(pa.int32())` | `object` (lista de enteros) |
| `pa.struct([('campo1', pa.int32()), ('campo2', pa.string())])` | `object` (registro con campos) |

La conversión de *PyArrow* a *Pandas* es directa mediante el método `df.astype()` sin pérdida de información, aunque algunos tipos complejos se representan como objetos genéricos en *Pandas*.


**Ejemplo:**

* La siguiente celda crea un *DataFrame* de *Pandas* y convierte columnas seleccionadas al tipo `pd.ArrowDtype`, activando *PyArrow* como *backend* para esas columnas.

In [ ]:
df_arrow = pd.DataFrame(
    {
        'id': [1, 2, 3, 4, 5],
        'nombre': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
        'edad': [25, 30, 35, 28, 32]
    }
)

df_arrow['nombre'] = df_arrow['nombre'].astype(pd.ArrowDtype(pa.string()))
df_arrow['edad'] = df_arrow['edad'].astype(pd.ArrowDtype(pa.int32()))

* La siguiente celda llama a `.info()` para verificar que las columnas `nombre` y `edad` tienen tipos *PyArrow*.

In [ ]:
df_arrow.info()

* La siguiente celda compara el consumo de memoria de un *DataFrame* con tipos *NumPy* (por defecto en *Pandas*) contra uno con tipos *PyArrow* usando `memory_usage(deep=True)`.

In [ ]:
df_normal = pd.DataFrame({'col': np.arange(1000000)})
mem_normal = df_normal.memory_usage(deep=True).sum() / 1024**2

df_arrow = df_normal.copy()
df_arrow['col'] = df_arrow['col'].astype(pd.ArrowDtype(pa.int64()))
mem_arrow = df_arrow.memory_usage(deep=True).sum() / 1024**2

print(f"Memoria NumPy (normal): {mem_normal:.2f} MB")
print(f"Memoria PyArrow: {mem_arrow:.2f} MB")
print(f"Reducción: {(1 - mem_arrow/mem_normal) * 100:.1f}%")

## Formatos Soportados.

*PyArrow* soporta lectura y escritura de múltiples formatos binarios optimizados.

**Ejemplos:**

* La siguiente ruta creará el directorio definido en `ruta` si no existe.

In [ ]:
dir = 'data_tmp'
os.makedirs(dir, exist_ok=True)

* La siguiente celda crea una *table* de ejemplo, la escribe en formato *Parquet* con `pq.write_table()` y la lee de vuelta con `pq.read_table()`, demostrando el ciclo completo de serialización.

In [ ]:
parquet_file_url = os.path.join(dir, 'tabla_ejemplo.parquet')
data = {
    'id': list(range(1, 11)),
    'valor': np.random.rand(10),
    'categoría': ['A', 'B', 'A', 'C', 'B'] * 2
}
table = pa.table(data)

import pyarrow.parquet as pq
pq.write_table(table, parquet_file_url)
print("✓ Guardado en Parquet")

tabla_leida = pq.read_table(parquet_file_url)
print(f"✓ Leído de Parquet:\n{tabla_leida}")

* La siguiente celda verificará la existencia del archivo *Parquet* creado usando `os.path.exists()`.

In [ ]:
os.path.exists(parquet_file_url)

* La siguiente celda lee los metadatos del archivo *Parquet* usando `pq.ParquetFile()` sin cargar los datos en memoria: *schema*, número de filas y columnas.

In [ ]:
parquet_file = pq.ParquetFile(parquet_file_url)
print(f"Schema: {parquet_file.schema}")
print(f"Número de filas: {parquet_file.metadata.num_rows}")
print(f"Número de columnas: {parquet_file.metadata.num_columns}")

* La siguiente celda leerá el archivo `datos_ejemplo.csv` con `pyarrow.csv.read_csv()` y devolverá una `Table` de *PyArrow*, demostrando que la lectura de *CSV* con *PyArrow* es directamente compatible sin pasar por *Pandas*.

In [ ]:
import pyarrow.csv as csv

csv_file_url = os.path.join(dir, 'tabla_ejemplo.csv')

df_ejemplo = pd.DataFrame({
    'id': range(1, 6),
    'nombre': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'valor': [10.5, 20.3, 15.7, 18.2, 22.1]
})
df_ejemplo.to_csv(csv_file_url, index=False)

tabla_csv = csv.read_csv(csv_file_url)
print(f"Tabla desde CSV:\n{tabla_csv}")

* La siguiente celda mostrará que el archivo `datos_ejemplo.csv` existe usando `os.path.exists()`.

In [ ]:
os.path.exists(csv_file_url)

## Interoperabilidad con *Polars*.

*Polars* es una biblioteca de análisis de datos de alto rendimiento que utiliza el formato *Arrow* como representación interna de memoria. Gracias a esta base común, la conversión entre una `Table` de *PyArrow* y un `DataFrame` de *Polars* es prácticamente *zero-copy*: no se copian los datos, solo se transfiere la referencia al bloque de memoria.

> **Lazy evaluation** (evaluación perezosa) significa que las operaciones sobre los datos no se ejecutan al definirlas, sino que se acumulan en un **plan de consulta**. Ese plan se optimiza y ejecuta solo cuando el resultado es solicitado explícitamente. Esto permite al motor eliminar cálculos innecesarios, reordenar operaciones y reducir el uso de memoria. *Polars* hace uso intensivo de este mecanismo a través de su *API lazy*.

*Polars* se estudia en detalle a partir del siguiente notebook.

**Ejemplo:**

* La siguiente celda convierte una `Table` de *PyArrow* a un `DataFrame` de *Polars* con `pl.from_arrow()`, sin copiar los datos subyacentes.

In [ ]:
import polars as pl

table_pa = pa.table({
    'nombre':  ['Alice', 'Bob', 'Charlie'],
    'edad':    [25, 30, 35],
    'salario': [50000.0, 60000.0, 75000.0],
})

df_polars = pl.from_arrow(table_pa)
df_polars

In [ ]:
df_polars.schema

* La siguiente celda convierte el *dataframe* de *Polars* de vuelta a una `Table` de *PyArrow* con `.to_arrow()`, verificando que el *schema* se preserva.

In [ ]:
table_recuperada = df_polars.to_arrow()
table_recuperada

In [ ]:
table_recuperada.schema

## Resumen.

*PyArrow* proporciona las bases del ecosistema moderno de datos en *Python*:

| Concepto | Descripción |
|---|---|
| **Arrays** | Colecciones homogéneas almacenadas en formato columnar eficiente |
| **Tables** | Conjuntos de *arrays* con *schema* definido, equivalente a un *DataFrame* |
| **Schema** | Definición y validación estricta de tipos de datos |
| **Tipos complejos** | Soporte nativo para listas, structs y datos semiestructurados |
| **Parquet** | Formato binario columnar, estándar para intercambio de datos a gran escala |

### Relación con *Polars*.

*Polars* utiliza el formato *Arrow* como su representación interna de memoria. Esto significa que:

* La transferencia de datos entre *PyArrow* y *Polars* es *zero-copy* en la mayoría de los casos.
* Los archivos *Parquet* escritos con *PyArrow* son directamente legibles por *Polars* y viceversa.
* El *schema* de *PyArrow* es equivalente al *schema* de *Polars*.

En los próximos capítulos se verá cómo *Polars* aprovecha esta base para ofrecer un API de análisis de datos más expresivo y de mayor rendimiento que *Pandas*.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>